### Installing Semantic Link Labs

In [ ]:
%pip install semantic-link-labs

### Importing packages

In [ ]:
import os
import sys
import zipfile
import requests
import xml.etree.ElementTree as ET
from pythonnet import load
import io
import zipfile
import pandas as pd
import re
import shutil 
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

### Configuration Variables

In [ ]:
# DAX Lib TMDL Pckage Name
package_name = "PowerofBI.IBCS" 
# Workspace, Semantic Models you want to deploy the Dax Lib Package to
model_pairs = [
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "21cc43f4-e89c-4b5f-b97d-af6abb9cebf6"),
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "51a3dfd7-4d14-4376-a3fe-4cc898eb5caa"),
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "7535e0df-63f3-4cb5-9e45-bf4ab75961cb")
]
# Fabric Lakehouse path for dll files
BIN_DIR = "/lakehouse/default/Files/bin/"
# Fabric Lakehouse path for tmdl files
EXPORT_BASE_PATH = "/lakehouse/default/Files/temp_tmdl_export"
#.net Package info
PACKAGE_ID = "DaxLib.Client"
MANDATORY_PACKAGES = ["System.IO.Packaging", "NuGet.Versioning", "Newtonsoft.Json", "NuGet.Frameworks"]

### Configuring Enviornment

In [ ]:
# CRITICAL: Must be set before 'import clr' or 'from pythonnet import load'
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
#Making bin Directory
os.makedirs(BIN_DIR, exist_ok=True)
#Making TMDL Director
os.makedirs(EXPORT_BASE_PATH, exist_ok=True)

if BIN_DIR not in sys.path:
    sys.path.append(BIN_DIR)

print(f"✅ Environment initialized. Bin directory: {BIN_DIR}. TMDL directory: {EXPORT_BASE_PATH}")

### Helper Functions

In [ ]:
#Fetches the latest stable version from NuGet API.
def get_latest_version(package_id):
    url = f"https://api.nuget.org/v3-flatcontainer/{package_id.lower()}/index.json"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()['versions'][-1]

#Downloads nupkg and extracts relevant DLLs to a directory.
def download_and_extract(BIN_DIR, package_id, version, extract_all=False):
    dll_path = os.path.join(BIN_DIR, f"{package_id}.dll")
    print(f"📥 Downloading {package_id} v{version}...")
    url = f"https://www.nuget.org/api/v2/package/{package_id}/{version}"
    zip_path = f"/tmp/{package_id}.nupkg"
    
    with open(zip_path, "wb") as f:
        f.write(requests.get(url, timeout=30).content)
    
    dependencies = []
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Extract DLLs
        for file_info in z.infolist():
            # Sometimes .dll can be .DLL
            if ".dll" in file_info.filename.lower() and "lib/" in file_info.filename:
                # Target modern runtimes
                if extract_all or any(fw in file_info.filename for fw in ["net9.0", "net8.0", "net6.0", "netstandard2.1"]):
                    dll_name = os.path.basename(file_info.filename)
                    with open(os.path.join(BIN_DIR, dll_name), "wb") as f:
                        f.write(z.read(file_info.filename))
                    print(f"   ✅ Extracted: {dll_name}")
        
        # Parse dependencies from .nuspec if this is the main package
        if extract_all:
            nuspec = next(f for f in z.namelist() if f.endswith('.nuspec'))
            root = ET.fromstring(z.read(nuspec))
            ns = {'ns': 'http://schemas.microsoft.com/packaging/2012/06/nuspec.xsd'}
            dependencies = [(d.get('id'), d.get('version')) for d in root.findall(".//ns:dependency", ns)]
            
    return dependencies

#Initalizes CoreCLR and installs DAXLib Client + System.IO.Packaging
def init_clr_and_refs(bin_path):
    if "clr" not in sys.modules:
        from pythonnet import load
        load("coreclr")
        print("✅ .NET CoreCLR Initialized.")
    
    import clr
    from System import AppDomain
    
    assemblies = [a.GetName().Name for a in AppDomain.CurrentDomain.GetAssemblies()]
    
    required_dlls = {
        "System.IO.Packaging": os.path.join(bin_path, "System.IO.Packaging.dll"),
        "DaxLib.Client": os.path.join(bin_path, "DaxLib.Client.dll")
    }
    
    for name, path in required_dlls.items():
        if name not in assemblies:
            try:
                clr.AddReference(path)
                print(f"✅ Reference added: {name}")
            except Exception as e:
                print(f"❌ Failed to add reference {name}: {e}")
        else:
            print(f"ℹ️ {name} is already referenced.")

#Parses the TMDL Text Block rturns a dataframe of functions
def parse_tmdl_by_indentation(tmdl_text):
    lines = tmdl_text.splitlines()
    functions = []
    
    current_name = None
    current_expression = []
    is_capturing = False
    function_indent_level = None

    for line in lines:
        # Remove markdown code fence characters but keep the line
        cleaned_line = line.replace("```", "")
        
        # Check for function definition on the cleaned line
        if cleaned_line.strip().startswith("function '"):
            if current_name and current_expression:
                functions.append({
                    "name": current_name,
                    "expression": "\n".join(current_expression).strip()
                })
            
            match = re.search(r"function\s+'([^']+)'", cleaned_line)
            if match:
                current_name = match.group(1)
                current_expression = []
                is_capturing = True
                function_indent_level = None  # Reset indent level
            continue

        if is_capturing:
            # Use cleaned line for processing
            line = cleaned_line
            
            # Empty lines are always included
            if line.strip() == "":
                current_expression.append(line)
            else:
                # Calculate indentation level
                indent = len(line) - len(line.lstrip())
                
                # Set the function body indent level from the first non-empty line
                if function_indent_level is None:
                    function_indent_level = indent
                    current_expression.append(line)
                # If indentation is less than function body level, stop capturing
                elif indent < function_indent_level:
                    functions.append({
                        "name": current_name,
                        "expression": "\n".join(current_expression).strip()
                    })
                    current_name = None
                    current_expression = []
                    is_capturing = False
                    function_indent_level = None
                # Otherwise, it's part of the function body
                else:
                    current_expression.append(line)

    if current_name and current_expression:
        functions.append({
            "name": current_name,
            "expression": "\n".join(current_expression).strip()
        })

    return functions
#Deletes Directories
def cleanup_directories(paths_to_remove):
    if os.path.exists(paths_to_remove):
        try:
            # shutil.rmtree deletes a directory and all its contents (files/subfolders)
            shutil.rmtree(paths_to_remove)
            print(f"🗑️ Successfully deleted directory: {paths_to_remove}")
        except Exception as e:
            print(f"⚠️ Could not delete directory {paths_to_remove}: {e}")


### Downloads DAX Lib Client Files to Lakehouse

In [ ]:
#gets the most recent version of DAX Lib, then downloads and extracts dlls
print(f"🔍 Resolving dependencies for {PACKAGE_ID}...")
latest_ver = get_latest_version(PACKAGE_ID)
deps = download_and_extract(BIN_DIR, PACKAGE_ID, latest_ver, extract_all=True)

#gets the most recent version of the mandatory packages, then downloads and extracts dlls
for pkg in MANDATORY_PACKAGES:
    deps.append((pkg, get_latest_version(pkg)))

seen = set()
for pkg_id, pkg_ver in deps:
    if pkg_id not in seen:
        seen.add(pkg_id)
        try:
            download_and_extract(BIN_DIR, pkg_id, pkg_ver)
        except Exception as e:
            print(f"   ⚠️ Could not download {pkg_id}: {e}")

print("\n✨ All dependencies processed.")

### Downloads TMDL to Lakehouse

In [ ]:
try:
    init_clr_and_refs(BIN_DIR)
    
    from DaxLib.Client import DaxLibClient, ProductInformation
    from DaxLib.Packaging import PackageId
    from System.Threading import CancellationToken
    
    print(f"✅ DaxLib client Loaded!")
    
    client = DaxLibClient(ProductInformation("FabricNotebook", "1.0"))
    token = CancellationToken.get_None()
    
    #Fetchs latest version of the DAX UDF package 
    pkg_id = PackageId(package_name)
    versions = list(client.GetPackageVersionsAsync(pkg_id, token).Result)
    latest = versions[0]
    print(f"📦 Target: {package_name} (Latest: {latest.ToString()})")

    #Downloads the DAX UDF package
    stream = client.DownloadPackageAsync(pkg_id, latest, token).Result
    stream.Position = 0
    bytes_data = stream.ToArray()

    #Looks for TMDL files in the package and Saves them to the tmdl export pack
    with zipfile.ZipFile(io.BytesIO(bytes_data), 'r') as z:
        tmdl_files = [n for n in z.namelist() if n.endswith('.tmdl') or '/tmdl/' in n.lower()]
        
        print(f"\n{'='*40}\nEXPORTING FLAT TMDL TO LAKEHOUSE\n{'='*40}")
        
        for name in tmdl_files:
            filename_only = os.path.basename(name)
            if not filename_only:
                continue
                
            target_path = os.path.join(EXPORT_BASE_PATH, filename_only)
            
            content = z.read(name) 
            with open(target_path, "wb") as f:
                f.write(content)
            
            print(f"💾 Saved to Root: {filename_only}")

    stream.Close()
    print(f"\n✅ Operation Complete. All files are in the root of: {EXPORT_BASE_PATH}")

except Exception as e:
    print(f"❌ Error during execution: {e}")

### Parses TMDL functions into Dataframe and Cleans up

In [ ]:

all_parsed_functions = []

if os.path.exists(EXPORT_BASE_PATH):
    #Finds all TMDL files in the export directory and process all of them into a dataframe
    tmdl_files = [f for f in os.listdir(EXPORT_BASE_PATH) if f.endswith('.tmdl')]
    
    if not tmdl_files:
        print("⚠️ No TMDL files found in the directory.")
    else:
        for file_name in tmdl_files:
            file_path = os.path.join(EXPORT_BASE_PATH, file_name)
            print(f"📖 Processing: {file_name}")
            
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                file_functions = parse_tmdl_by_indentation(content)
                all_parsed_functions.extend(file_functions)

    #Creates a DataFrame of the functions
    df_tmdl = pd.DataFrame(all_parsed_functions)
    
    print(f"\n✅ Successfully parsed {len(df_tmdl)} functions.")
else:
    print(f"❌ Path not found: {EXPORT_BASE_PATH}")

cleanup_directories(BIN_DIR)
cleanup_directories(EXPORT_BASE_PATH)

In [ ]:
display(df_tmdl)

### Deploys functions using Semantic Link Labs to Semantic Models

In [ ]:
print(f"🚀 Starting deployment of {len(df_tmdl)} functions to {len(model_pairs)} models...\n")

for workspace_id, dataset_id in model_pairs:
    print(f"📂 Connecting to Workspace: {workspace_id} | Model: {dataset_id}")
    
    try:
        # Open connection to the semantic model
        with connect_semantic_model(dataset=dataset_id, readonly=False, workspace=workspace_id) as tom:
            
            for index, row in df_tmdl.iterrows():
                func_name = row['name']
                func_expression = row['expression']
                
                # Deploy the function to the current model
                tom.set_user_defined_function(name=func_name, expression=func_expression)
                print(f"   ✅ Deployed: {func_name}")
                
        print(f"✨ Finished deployment for model: {dataset_id}\n")
        
    except Exception as e:
        print(f"❌ Failed to deploy to model {dataset_id}: {e}")

print("🏁 All deployments complete.")